# 面试问题：Text-to-SQL Agent 怎样实现 schema linking、查询约束与安全执行？

**一句话回答。** 模型只能提出结构化查询计划；确定性控制面负责 schema linking、只读语法、表/列白名单、租户谓词、参数绑定、行数/成本限制和执行后验证。语法合法不代表语义正确，也绝不等于允许执行写操作。

本 Notebook 用 Python 标准库手写最小数据合同、状态机、验证器和失败分支。断言针对受控小数据，不等于模型语义正确、数据库安全、图谱质量或生产 Agent 的安全保证。

**资料入口。** [PICARD](https://arxiv.org/abs/2109.05093) 通过增量解析约束形式语言解码；本例把受约束输出进一步接入数据库权限与租户控制。


In [ ]:
question = "Text-to-SQL 安全执行"  # 执行本行的状态、计算或校验逻辑。
assert "SQL" in question  # 执行本行的状态、计算或校验逻辑。
assert 2 + 3 == 5  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 数据库 schema 是版本化接口，而不是 prompt 里的几行文字

每个表、列、类型、可读角色和 schema revision 都应从可信元数据读取。模型看到的 schema 摘要不拥有权限；实际 SQL 只能引用允许的对象，schema 漂移时需拒绝旧计划或重做 linking。


In [ ]:
schema = {"orders": {"columns": ("id", "tenant_id", "status", "amount"), "read_roles": ("support",), "revision": "db-v1"}}  # 执行本行的状态、计算或校验逻辑。
rows = [{"id": "o1", "tenant_id": "t1", "status": "refunded", "amount": 30}, {"id": "o2", "tenant_id": "t2", "status": "paid", "amount": 50}]  # 执行本行的状态、计算或校验逻辑。
assert schema["orders"]["revision"] == "db-v1"  # 执行本行的状态、计算或校验逻辑。
assert "tenant_id" in schema["orders"]["columns"]  # 执行本行的状态、计算或校验逻辑。
assert len(rows) == 2  # 执行本行的状态、计算或校验逻辑。

## 2. schema linking 只产出候选，不直接执行

教学用关键词映射模拟模型/检索器给出的 table-column 候选。真实系统可用 embedding、catalog 检索或 constrained decoding，但要记录候选、置信度和 schema revision，不能把自然语言直接拼到 SQL。


In [ ]:
links = {"退款": ("orders", "status"), "订单": ("orders", "id"), "金额": ("orders", "amount")}  # 执行本行的状态、计算或校验逻辑。
def schema_link(text):  # 执行本行的状态、计算或校验逻辑。
    return [value for key, value in links.items() if key in text]  # 执行本行的状态、计算或校验逻辑。
candidates = schema_link("查询退款订单金额")  # 执行本行的状态、计算或校验逻辑。
assert ("orders", "status") in candidates  # 执行本行的状态、计算或校验逻辑。
assert ("orders", "amount") in candidates  # 执行本行的状态、计算或校验逻辑。
assert all(table == "orders" for table, column in candidates)  # 执行本行的状态、计算或校验逻辑。

## 3. 模型输出受限为查询计划 AST

不接受自由 SQL 字符串作为执行输入。计划必须明确 select、from、where 和 limit；控制面验证表/列、谓词和上限后才可编译。复杂 join、聚合和子查询应有单独的 grammar 与成本估计。


In [ ]:
plan = {"select": ("id", "status", "amount"), "from": "orders", "where": {"tenant_id": "t1", "status": "refunded"}, "limit": 20, "schema_revision": "db-v1"}  # 执行本行的状态、计算或校验逻辑。
def valid_plan(plan_value, role):  # 执行本行的状态、计算或校验逻辑。
    meta = schema.get(plan_value["from"])  # 执行本行的状态、计算或校验逻辑。
    return meta is not None and role in meta["read_roles"] and set(plan_value["select"]).issubset(meta["columns"]) and set(plan_value["where"]).issubset(meta["columns"]) and "tenant_id" in plan_value["where"] and 0 < plan_value["limit"] <= 100  # 执行本行的状态、计算或校验逻辑。
assert valid_plan(plan, "support")  # 执行本行的状态、计算或校验逻辑。
assert not valid_plan({**plan, "limit": 101}, "support")  # 执行本行的状态、计算或校验逻辑。
assert not valid_plan({**plan, "select": ("password",)}, "support")  # 执行本行的状态、计算或校验逻辑。

## 4. 编译器只生成参数化、只读 SQL

参数值进入绑定字典而不是字符串插值；编译器本身固定 SELECT 模板，也不允许模型给出分号、函数名或 DDL/DML。数据库账户还应是只读、按租户隔离，不能只依赖应用层检查。


In [ ]:
def compile_select(plan_value):  # 执行本行的状态、计算或校验逻辑。
    selected = ", ".join(plan_value["select"])  # 执行本行的状态、计算或校验逻辑。
    predicates = " AND ".join(column + " = :" + column for column in plan_value["where"])  # 执行本行的状态、计算或校验逻辑。
    statement = "SELECT " + selected + " FROM " + plan_value["from"] + " WHERE " + predicates + " LIMIT " + str(plan_value["limit"])  # 执行本行的状态、计算或校验逻辑。
    return statement, dict(plan_value["where"])  # 执行本行的状态、计算或校验逻辑。
statement, params = compile_select(plan)  # 执行本行的状态、计算或校验逻辑。
assert statement.startswith("SELECT id, status, amount FROM orders")  # 执行本行的状态、计算或校验逻辑。
assert ":tenant_id" in statement and ":status" in statement  # 执行本行的状态、计算或校验逻辑。
assert params == {"tenant_id": "t1", "status": "refunded"}  # 执行本行的状态、计算或校验逻辑。

## 5. 执行前后都做租户和结果门禁

本例用内存行模拟数据库执行。真实系统在数据库侧使用 row-level security/视图，并限制 timeout、扫描字节和返回行数；执行结果还要被标记来源和查询计划，供答案引用或审计。


In [ ]:
def execute(plan_value, data):  # 执行本行的状态、计算或校验逻辑。
    return [row for row in data if all(row[column] == value for column, value in plan_value["where"].items())][:plan_value["limit"]]  # 执行本行的状态、计算或校验逻辑。
result = execute(plan, rows)  # 执行本行的状态、计算或校验逻辑。
assert result == [{"id": "o1", "tenant_id": "t1", "status": "refunded", "amount": 30}]  # 执行本行的状态、计算或校验逻辑。
assert all(row["tenant_id"] == "t1" for row in result)  # 执行本行的状态、计算或校验逻辑。
assert len(result) <= plan["limit"]  # 执行本行的状态、计算或校验逻辑。

## 6. 拒绝注入、写操作与跨租户计划

安全测试应包含模型被诱导输出危险语句、带分号的表名、遗漏 tenant predicate 和越权角色。即便 SQL parser 认为语法正确，业务策略不满足也必须拒绝，而不是自动修补成另一个语义。


In [ ]:
unsafe = {**plan, "from": "orders; DROP TABLE orders"}  # 执行本行的状态、计算或校验逻辑。
cross_tenant = {**plan, "where": {"status": "refunded"}}  # 执行本行的状态、计算或校验逻辑。
assert not valid_plan(unsafe, "support")  # 执行本行的状态、计算或校验逻辑。
assert not valid_plan(cross_tenant, "support")  # 执行本行的状态、计算或校验逻辑。
assert not valid_plan(plan, "viewer")  # 执行本行的状态、计算或校验逻辑。

## 7. schema 变更与计划重放必须显式不兼容

缓存的 query plan、few-shot 示例和 golden SQL 都要绑定 schema revision。列改名或权限变更后，应重新 linking 并人工审查回归；把旧计划悄悄映射到新列会造成难发现的数据泄露或错误结论。


In [ ]:
def compatible(plan_value, current_revision):  # 执行本行的状态、计算或校验逻辑。
    return plan_value["schema_revision"] == current_revision  # 执行本行的状态、计算或校验逻辑。
assert compatible(plan, "db-v1")  # 执行本行的状态、计算或校验逻辑。
assert not compatible(plan, "db-v2")  # 执行本行的状态、计算或校验逻辑。
assert schema["orders"]["revision"] == plan["schema_revision"]  # 执行本行的状态、计算或校验逻辑。

## 8. 评测要分开 SQL、执行和业务答案

应同时测 schema linking recall、AST valid rate、execution accuracy、tenant leak、拒绝率、超时/成本和最终自然语言答案的证据一致性。只比较 SQL 字符串 exact match 会错罚等价查询，也会漏掉越权。


In [ ]:
def evidence_from_result(statement_value, rows_value):  # 执行本行的状态、计算或校验逻辑。
    return {"statement": statement_value, "row_ids": tuple(row["id"] for row in rows_value), "schema": plan["schema_revision"]}  # 执行本行的状态、计算或校验逻辑。
evidence = evidence_from_result(statement, result)  # 执行本行的状态、计算或校验逻辑。
assert evidence["row_ids"] == ("o1",)  # 执行本行的状态、计算或校验逻辑。
assert evidence["schema"] == "db-v1"  # 执行本行的状态、计算或校验逻辑。
assert "DROP" not in evidence["statement"]  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试回答应从“自然语言到受限 AST”开始，再说明 schema/角色/租户/参数/成本门禁、数据库侧 RLS、结果 provenance 和评测。Text-to-SQL 的困难不只在生成可执行 SELECT，更在确保它在正确 schema、正确权限和正确时间范围内执行。
